# Day 17 - Delta History Time Travel and Recovery

**Topic:** Delta History Time Travel and Recovery  
**Main environment:** Microsoft Fabric Notebook + Fabric Lakehouse  
**Focus:** เข้าใจว่า Delta table เก็บ version history อย่างไร และใช้ time travel / restore เพื่อ investigate และ recover จาก bad load ได้อย่างปลอดภัย

วันนี้เราจะฝึกสร้าง Delta table หลาย versions ใน Fabric Lakehouse, อ่าน table history, query ข้อมูลย้อนหลังด้วย `VERSION AS OF`, จำลอง bad overwrite และใช้ recovery mindset เพื่อกู้ table กลับไปยัง known-good version


## Goal of the day

1. อธิบายได้ว่า Delta table history และ table version คืออะไร
2. ใช้ `DESCRIBE HISTORY` เพื่อตรวจ transaction history ของ Delta table ได้
3. ใช้ time travel เช่น `VERSION AS OF` หรือ `timestampAsOf` เพื่ออ่านข้อมูลย้อนหลังได้
4. จำลอง bad overwrite แล้วตรวจสอบข้อมูลก่อนเกิด incident ได้
5. เข้าใจความต่างระหว่าง **time travel** กับ **restore / recovery** ในมุม production


## Why it matters in production

ใน production pipeline ข้อมูลอาจพังได้จากหลายสาเหตุ เช่น overwrite ผิด table, merge logic ผิด, source batch มีข้อมูลผิด, schema change หลุดมา หรือ dedup logic ทำให้ records หาย

Delta history และ time travel ช่วยให้เรา:

- ตรวจได้ว่า table ถูกเขียนด้วย operation อะไรบ้าง เช่น `WRITE`, `MERGE`, `RESTORE`
- ย้อนดูข้อมูลก่อนและหลัง pipeline run เพื่อหา root cause
- อ่าน snapshot เก่าก่อนตัดสินใจ rollback จริง
- ทำ recovery โดยอิง version ที่ตรวจสอบแล้ว ไม่ใช่เดาสุ่ม
- สร้างความเชื่อมั่นว่า Lakehouse table มี auditability มากกว่าการเก็บข้อมูลเป็นไฟล์อย่างเดียว

Production takeaway:

> ก่อน recover table ต้องรู้ก่อนว่า version ไหนคือ known-good version และต้อง validate ด้วย time travel ก่อนเปลี่ยน current table state


## Key concepts

| Concept | Meaning |
|---|---|
| Delta history | ประวัติ transaction ของ Delta table เช่น write, merge, overwrite, restore |
| Version | เลข snapshot ของ table หลังแต่ละ committed change เริ่มจาก version 0 |
| Commit | การเปลี่ยนแปลง table ที่ถูกบันทึกลง Delta transaction log |
| Time travel | การ query table ย้อนหลังด้วย version หรือ timestamp โดยไม่เปลี่ยน current table |
| `VERSION AS OF` | SQL syntax สำหรับอ่าน Delta table ณ version ที่กำหนด |
| `versionAsOf` | PySpark option สำหรับอ่าน Delta table ณ version ที่กำหนด |
| Known-good version | version ที่ตรวจแล้วว่าข้อมูลถูกต้องก่อนเกิด bad load |
| Restore | การทำให้ current table state กลับไปตรงกับ version/timestamp เก่า |
| Recovery mindset | ขั้นตอนคิดก่อนกู้ข้อมูล เช่น inspect history, validate old snapshot, restore เฉพาะเมื่อมั่นใจ |
| Retention / VACUUM | ตัวกำหนดว่า historical files ยังอยู่ให้ time travel หรือ restore ได้ไกลแค่ไหน |


## Concept explanation

### Delta history คืออะไร?

Delta table ไม่ได้มีแค่ Parquet data files แต่มี `_delta_log` ที่เก็บ transaction metadata ของ table ด้วย ทุกครั้งที่มีการ write, append, merge, overwrite หรือ restore จะเกิด table version ใหม่

พูดง่าย ๆ:

> Delta table = data files + transaction log  
> Transaction log = สิ่งที่ทำให้เรารู้ว่า table เคยอยู่ใน state ไหนบ้าง

### Time travel คืออะไร?

Time travel คือการอ่าน table ย้อนหลังโดยระบุ version หรือ timestamp เช่น:

```sql
SELECT * FROM table_name VERSION AS OF 3
```

หรือใน PySpark:

```python
spark.read.format("delta").option("versionAsOf", 3).table("table_name")
```

สำคัญมาก: time travel เป็นการอ่าน snapshot เก่าแบบ read-only ไม่ได้ทำให้ current table เปลี่ยนกลับไปเอง

### Restore / recovery ต่างจาก time travel ยังไง?

- **Time travel** = อ่าน version เก่าเพื่อ inspect / compare / validate
- **Restore** = เปลี่ยน current table state ให้กลับไปเหมือน version เก่า

Production flow ที่ปลอดภัยมักเป็น:

1. Run `DESCRIBE HISTORY`
2. หา version ก่อนเกิด bad load
3. Query version นั้นด้วย time travel
4. Validate row count, key records, schema และ business logic
5. ค่อย restore หรือ create recovery copy เมื่อมั่นใจ

### Retention สำคัญยังไง?

Time travel และ restore จะใช้ได้ก็ต่อเมื่อ Delta log และ physical data files ที่ version นั้นต้องการยังอยู่ ถ้า `VACUUM` ลบ files ที่ old version ต้องใช้ไปแล้ว การย้อนกลับไปที่ version นั้นจะทำไม่ได้


## Mermaid diagram: Delta History, Time Travel, and Recovery Flow

```mermaid
flowchart LR
    A[Version 0: Initial load] --> B[Version 1: Append new records]
    B --> C[Version 2: MERGE updates]
    C --> D[Version 3: Bad overwrite]

    H[DESCRIBE HISTORY] --> C
    C --> T[Time travel to known-good version]
    T --> V[Validate old snapshot]
    V --> R[RESTORE table or create recovery copy]
    R --> E[New current version after recovery]
```

Key idea:

> History ช่วยหา version, time travel ช่วยตรวจข้อมูลเก่า, restore/recovery ช่วยทำให้ current table กลับไปสู่ state ที่ถูกต้อง


## Setup / imports


In [1]:
from pyspark.sql import functions as F
from pyspark.sql import types as T

StatementMeta(, efcf106b-f584-4fbe-abb8-5abf1cab0427, 3, Finished, Available, Finished, False)

## Create mock data

In [2]:
target_table_name = "day17_customer_profile_delta"
recovery_copy_table_name = "day17_customer_profile_recovery_copy"

customer_schema = T.StructType([
    T.StructField("customer_id", T.StringType(), False),
    T.StructField("customer_name", T.StringType(), True),
    T.StructField("email", T.StringType(), True),
    T.StructField("status", T.StringType(), True),
    T.StructField("customer_score", T.IntegerType(), True),
    T.StructField("updated_at_raw", T.StringType(), True),
    T.StructField("source_system", T.StringType(), True),
])

initial_customer_data = [
    ("C001", "Anan", "anan@example.com", "active", 82, "2026-06-01 09:00:00", "crm"),
    ("C002", "Benjawan", "benjawan@example.com", "active", 76, "2026-06-01 09:05:00", "crm"),
    ("C003", "Chai", "chai@example.com", "inactive", 40, "2026-06-01 09:10:00", "crm"),
    ("C004", "Darika", "darika@example.com", "active", 91, "2026-06-01 09:15:00", "crm"),
]

df_initial_customers = (
    spark.createDataFrame(initial_customer_data, customer_schema)
    .withColumn("updated_at", F.to_timestamp("updated_at_raw"))
    .drop("updated_at_raw")
)

df_initial_customers.show(truncate=False)
df_initial_customers.printSchema()

StatementMeta(, efcf106b-f584-4fbe-abb8-5abf1cab0427, 5, Finished, Available, Finished, False)

+-----------+-------------+--------------------+--------+--------------+-------------+-------------------+
|customer_id|customer_name|email               |status  |customer_score|source_system|updated_at         |
+-----------+-------------+--------------------+--------+--------------+-------------+-------------------+
|C001       |Anan         |anan@example.com    |active  |82            |crm          |2026-06-01 09:00:00|
|C002       |Benjawan     |benjawan@example.com|active  |76            |crm          |2026-06-01 09:05:00|
|C003       |Chai         |chai@example.com    |inactive|40            |crm          |2026-06-01 09:10:00|
|C004       |Darika       |darika@example.com  |active  |91            |crm          |2026-06-01 09:15:00|
+-----------+-------------+--------------------+--------+--------------+-------------+-------------------+

root
 |-- customer_id: string (nullable = false)
 |-- customer_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- status:

## PySpark code examples

ใน section นี้จะไล่จากการสร้าง table version แรก → append → merge → inspect history → time travel → simulate bad overwrite → recover table กลับจาก known-good version


### Example 1: Write initial Delta table version

Cell นี้สร้าง Delta table version แรกด้วย `mode("overwrite")` และ `saveAsTable()`

หลัง write ครั้งแรก table จะมี version 0


In [22]:
(
    df_initial_customers
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(target_table_name)
)

spark.read.table(target_table_name).orderBy("customer_id").show(truncate=False)

StatementMeta(, efcf106b-f584-4fbe-abb8-5abf1cab0427, 25, Finished, Available, Finished, False)

+-----------+-------------+--------------------+--------+--------------+-------------+-------------------+
|customer_id|customer_name|email               |status  |customer_score|source_system|updated_at         |
+-----------+-------------+--------------------+--------+--------------+-------------+-------------------+
|C001       |Anan         |anan@example.com    |active  |82            |crm          |2026-06-01 09:00:00|
|C002       |Benjawan     |benjawan@example.com|active  |76            |crm          |2026-06-01 09:05:00|
|C003       |Chai         |chai@example.com    |inactive|40            |crm          |2026-06-01 09:10:00|
|C004       |Darika       |darika@example.com  |active  |91            |crm          |2026-06-01 09:15:00|
+-----------+-------------+--------------------+--------+--------------+-------------+-------------------+



### Example 2: Append a new batch to create another version

การ append batch ใหม่จะสร้าง table version ถัดไป โดยไม่ลบ records เดิม


In [23]:
append_customer_data = [
    ("C005", "Ekkarat", "ekkarat@example.com", "active", 68, "2026-06-01 10:00:00", "mobile_app"),
    ("C006", "Fah", "fah@example.com", "pending", 55, "2026-06-01 10:05:00", "mobile_app"),
]

df_append_customers = (
    spark.createDataFrame(append_customer_data, customer_schema)
    .withColumn("updated_at", F.to_timestamp("updated_at_raw"))
    .drop("updated_at_raw")
)

(
    df_append_customers
    .write
    .format("delta")
    .mode("append")
    .saveAsTable(target_table_name)
)

spark.read.table(target_table_name).orderBy("customer_id").show(truncate=False)

StatementMeta(, efcf106b-f584-4fbe-abb8-5abf1cab0427, 26, Finished, Available, Finished, False)

+-----------+-------------+--------------------+--------+--------------+-------------+-------------------+
|customer_id|customer_name|email               |status  |customer_score|source_system|updated_at         |
+-----------+-------------+--------------------+--------+--------------+-------------+-------------------+
|C001       |Anan         |anan@example.com    |active  |82            |crm          |2026-06-01 09:00:00|
|C002       |Benjawan     |benjawan@example.com|active  |76            |crm          |2026-06-01 09:05:00|
|C003       |Chai         |chai@example.com    |inactive|40            |crm          |2026-06-01 09:10:00|
|C004       |Darika       |darika@example.com  |active  |91            |crm          |2026-06-01 09:15:00|
|C005       |Ekkarat      |ekkarat@example.com |active  |68            |mobile_app   |2026-06-01 10:00:00|
|C006       |Fah          |fah@example.com     |pending |55            |mobile_app   |2026-06-01 10:05:00|
+-----------+-------------+----------

### Example 3: Run MERGE update to create another version

ใช้ `MERGE` เพื่อ update customer ที่มีอยู่ และ insert customer ใหม่ เหมือน incremental update pattern เบื้องต้น


In [24]:
update_customer_data = [
    ("C002", "Benjawan", "benjawan.new@example.com", "active", 88, "2026-06-01 11:00:00", "crm"),
    ("C003", "Chai", "chai@example.com", "active", 61, "2026-06-01 11:05:00", "crm"),
    ("C007", "Ganda", "ganda@example.com", "active", 73, "2026-06-01 11:10:00", "partner_api"),
]

df_customer_updates = (
    spark.createDataFrame(update_customer_data, customer_schema)
    .withColumn("updated_at", F.to_timestamp("updated_at_raw"))
    .drop("updated_at_raw")
)

df_customer_updates.createOrReplaceTempView("day17_customer_updates_vw")

merge_sql = f'''
MERGE INTO {target_table_name} AS target
USING day17_customer_updates_vw AS source
ON target.customer_id = source.customer_id
WHEN MATCHED AND source.updated_at > target.updated_at THEN UPDATE SET
    target.customer_name = source.customer_name,
    target.email = source.email,
    target.status = source.status,
    target.customer_score = source.customer_score,
    target.updated_at = source.updated_at,
    target.source_system = source.source_system
WHEN NOT MATCHED THEN INSERT (
    customer_id,
    customer_name,
    email,
    status,
    customer_score,
    updated_at,
    source_system
) VALUES (
    source.customer_id,
    source.customer_name,
    source.email,
    source.status,
    source.customer_score,
    source.updated_at,
    source.source_system
)
'''

spark.sql(merge_sql)

spark.read.table(target_table_name).orderBy("customer_id").show(truncate=False)

StatementMeta(, efcf106b-f584-4fbe-abb8-5abf1cab0427, 27, Finished, Available, Finished, False)

+-----------+-------------+------------------------+-------+--------------+-------------+-------------------+
|customer_id|customer_name|email                   |status |customer_score|source_system|updated_at         |
+-----------+-------------+------------------------+-------+--------------+-------------+-------------------+
|C001       |Anan         |anan@example.com        |active |82            |crm          |2026-06-01 09:00:00|
|C002       |Benjawan     |benjawan.new@example.com|active |88            |crm          |2026-06-01 11:00:00|
|C003       |Chai         |chai@example.com        |active |61            |crm          |2026-06-01 11:05:00|
|C004       |Darika       |darika@example.com      |active |91            |crm          |2026-06-01 09:15:00|
|C005       |Ekkarat      |ekkarat@example.com     |active |68            |mobile_app   |2026-06-01 10:00:00|
|C006       |Fah          |fah@example.com         |pending|55            |mobile_app   |2026-06-01 10:05:00|
|C007     

### Example 4: Inspect table history

`DESCRIBE HISTORY` ช่วยให้เห็นว่า Delta table มี operations อะไรเกิดขึ้นบ้าง และแต่ละ operation สร้าง version ไหน

ใน production เราใช้ history เพื่อหา version ก่อนเกิด incident เช่น ก่อน bad overwrite หรือ bad merge


In [25]:
df_history = spark.sql(f"DESCRIBE HISTORY {target_table_name}")

(
    df_history
    .select(
        "version",
        "timestamp",
        "operation",
        "operationParameters",
        "isolationLevel"
    )
    .orderBy(F.col("version").desc())
    .show(truncate=False)
)

StatementMeta(, efcf106b-f584-4fbe-abb8-5abf1cab0427, 28, Finished, Available, Finished, False)

+-------+-----------------------+---------------------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------------+
|version|timestamp              |operation                        |operationParameters                                                                                                                                                                                                                                  |isolationLevel|
+-------+-----------------------+---------------------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------------+
|2      |2026

### Example 5: Read older table versions with time travel

ใช้ `versionAsOf` เพื่ออ่าน snapshot ย้อนหลังโดยไม่เปลี่ยน current table

ตัวอย่างนี้อ่าน version 0 ซึ่งเป็น initial load แล้วเทียบกับ current table


In [26]:
df_version_0 = (
    spark.read
    .format("delta")
    .option("versionAsOf", 0)
    .table(target_table_name)
)

print("Version 0 snapshot:")
df_version_0.orderBy("customer_id").show(truncate=False)

print("Current table snapshot:")
spark.read.table(target_table_name).orderBy("customer_id").show(truncate=False)

StatementMeta(, efcf106b-f584-4fbe-abb8-5abf1cab0427, 29, Finished, Available, Finished, False)

Version 0 snapshot:
+-----------+-------------+--------------------+--------+--------------+-------------+-------------------+
|customer_id|customer_name|email               |status  |customer_score|source_system|updated_at         |
+-----------+-------------+--------------------+--------+--------------+-------------+-------------------+
|C001       |Anan         |anan@example.com    |active  |82            |crm          |2026-06-01 09:00:00|
|C002       |Benjawan     |benjawan@example.com|active  |76            |crm          |2026-06-01 09:05:00|
|C003       |Chai         |chai@example.com    |inactive|40            |crm          |2026-06-01 09:10:00|
|C004       |Darika       |darika@example.com  |active  |91            |crm          |2026-06-01 09:15:00|
+-----------+-------------+--------------------+--------+--------------+-------------+-------------------+

Current table snapshot:
+-----------+-------------+------------------------+-------+--------------+-------------+----------

### Example 6: Capture the latest known-good version before a bad load

ก่อนจำลอง bad overwrite เราจะจำ latest version ปัจจุบันไว้เป็น `known_good_version`

ใน production ขั้นตอนนี้ควรมาจากการตรวจ history + validation ไม่ใช่เลือก version แบบเดาสุ่ม


In [27]:
known_good_version = (
    spark.sql(f"DESCRIBE HISTORY {target_table_name}")
    .agg(F.max("version").alias("latest_version"))
    .first()["latest_version"]  # first() returns the first metadata row as a Python Row object.
)

known_good_timestamp = (
    spark.sql(f"DESCRIBE HISTORY {target_table_name}")
    .filter(F.col("version") == known_good_version)
    .select("timestamp")
    .first()["timestamp"]
)

print("Known-good version before bad load:", known_good_version)
print("Known-good timestamp before bad load:", known_good_timestamp)

StatementMeta(, efcf106b-f584-4fbe-abb8-5abf1cab0427, 30, Finished, Available, Finished, False)

Known-good version before bad load: 2
Known-good timestamp before bad load: 2026-06-02 14:01:12.383000


### Example 7: Simulate a bad overwrite

Cell นี้ตั้งใจ overwrite table ด้วย bad batch ที่มีข้อมูลผิด เพื่อจำลอง incident

ในงานจริงนี่คือเหตุการณ์ที่อาจเกิดจาก write mode ผิด, source batch ผิด หรือ pipeline logic ผิด


In [28]:
bad_customer_data = [
    ("BAD001", "Unknown", "unknown@example.com", "unknown", -1, "2026-06-01 12:00:00", "broken_job"),
]

df_bad_customers = (
    spark.createDataFrame(bad_customer_data, customer_schema)
    .withColumn("updated_at", F.to_timestamp("updated_at_raw"))
    .drop("updated_at_raw")
)

(
    df_bad_customers
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(target_table_name)
)

print("Current table after bad overwrite:")
spark.read.table(target_table_name).show(truncate=False)

StatementMeta(, efcf106b-f584-4fbe-abb8-5abf1cab0427, 31, Finished, Available, Finished, False)

Current table after bad overwrite:
+-----------+-------------+-------------------+-------+--------------+-------------+-------------------+
|customer_id|customer_name|email              |status |customer_score|source_system|updated_at         |
+-----------+-------------+-------------------+-------+--------------+-------------+-------------------+
|BAD001     |Unknown      |unknown@example.com|unknown|-1            |broken_job   |2026-06-01 12:00:00|
+-----------+-------------+-------------------+-------+--------------+-------------+-------------------+



### Example 8: Use time travel to inspect data before the bad overwrite

ถึง current table จะถูก overwrite พังไปแล้ว แต่เรายังอ่าน known-good version ได้ด้วย time travel ถ้า historical files ยังอยู่


In [29]:
df_known_good_snapshot = (
    spark.read
    .format("delta")
    .option("versionAsOf", known_good_version)
    .table(target_table_name)
)

print("Known-good snapshot from time travel:")
df_known_good_snapshot.orderBy("customer_id").show(truncate=False)

current_count = spark.read.table(target_table_name).count()
known_good_count = df_known_good_snapshot.count()

print("Current row count after bad overwrite:", current_count)
print("Known-good row count before bad overwrite:", known_good_count)

StatementMeta(, efcf106b-f584-4fbe-abb8-5abf1cab0427, 32, Finished, Available, Finished, False)

Known-good snapshot from time travel:
+-----------+-------------+------------------------+-------+--------------+-------------+-------------------+
|customer_id|customer_name|email                   |status |customer_score|source_system|updated_at         |
+-----------+-------------+------------------------+-------+--------------+-------------+-------------------+
|C001       |Anan         |anan@example.com        |active |82            |crm          |2026-06-01 09:00:00|
|C002       |Benjawan     |benjawan.new@example.com|active |88            |crm          |2026-06-01 11:00:00|
|C003       |Chai         |chai@example.com        |active |61            |crm          |2026-06-01 11:05:00|
|C004       |Darika       |darika@example.com      |active |91            |crm          |2026-06-01 09:15:00|
|C005       |Ekkarat      |ekkarat@example.com     |active |68            |mobile_app   |2026-06-01 10:00:00|
|C006       |Fah          |fah@example.com         |pending|55            |mobile_

### Example 9: Optional timestamp-based time travel

`timestampAsOf` ใช้อ่าน table ตาม timestamp ได้เช่นกัน แต่ใน lab นี้ใช้ `versionAsOf` เป็นตัวหลัก เพราะระบุ snapshot ได้ตรงกว่าและอธิบายง่ายกว่า

In [31]:
df_known_good_by_timestamp = (
    spark.read
    .format("delta")
    .option("timestampAsOf", known_good_timestamp)
    .table(target_table_name)
)

print("Known-good snapshot from timestampAsOf:")
df_known_good_by_timestamp.orderBy("customer_id").show(truncate=False)

StatementMeta(, efcf106b-f584-4fbe-abb8-5abf1cab0427, 34, Finished, Available, Finished, False)

Known-good snapshot from timestampAsOf:
+-----------+-------------+------------------------+-------+--------------+-------------+-------------------+
|customer_id|customer_name|email                   |status |customer_score|source_system|updated_at         |
+-----------+-------------+------------------------+-------+--------------+-------------+-------------------+
|C001       |Anan         |anan@example.com        |active |82            |crm          |2026-06-01 09:00:00|
|C002       |Benjawan     |benjawan.new@example.com|active |88            |crm          |2026-06-01 11:00:00|
|C003       |Chai         |chai@example.com        |active |61            |crm          |2026-06-01 11:05:00|
|C004       |Darika       |darika@example.com      |active |91            |crm          |2026-06-01 09:15:00|
|C005       |Ekkarat      |ekkarat@example.com     |active |68            |mobile_app   |2026-06-01 10:00:00|
|C006       |Fah          |fah@example.com         |pending|55            |mobil

### Example 10: Restore table to the known-good version

หลังจาก validate แล้วว่า `known_good_version` ถูกต้อง เราสามารถ restore current table กลับไปยัง version นั้นได้

สำคัญ: `RESTORE` ไม่ใช่การลบ history แต่จะสร้าง version ใหม่ที่ทำให้ current state กลับไปตรงกับ version เก่า


In [32]:
spark.sql(f"RESTORE TABLE {target_table_name} TO VERSION AS OF {known_good_version}")

print("Current table after restore:")
spark.read.table(target_table_name).orderBy("customer_id").show(truncate=False)

print("History after restore:")
(
    spark.sql(f"DESCRIBE HISTORY {target_table_name}")
    .select("version", "timestamp", "operation", "operationParameters")
    .orderBy(F.col("version").desc())
    .show(truncate=False)
)

StatementMeta(, efcf106b-f584-4fbe-abb8-5abf1cab0427, 35, Finished, Available, Finished, False)

Current table after restore:
+-----------+-------------+------------------------+-------+--------------+-------------+-------------------+
|customer_id|customer_name|email                   |status |customer_score|source_system|updated_at         |
+-----------+-------------+------------------------+-------+--------------+-------------+-------------------+
|C001       |Anan         |anan@example.com        |active |82            |crm          |2026-06-01 09:00:00|
|C002       |Benjawan     |benjawan.new@example.com|active |88            |crm          |2026-06-01 11:00:00|
|C003       |Chai         |chai@example.com        |active |61            |crm          |2026-06-01 11:05:00|
|C004       |Darika       |darika@example.com      |active |91            |crm          |2026-06-01 09:15:00|
|C005       |Ekkarat      |ekkarat@example.com     |active |68            |mobile_app   |2026-06-01 10:00:00|
|C006       |Fah          |fah@example.com         |pending|55            |mobile_app   |20

### Example 11: Create a recovery copy for safe investigation

บางครั้งใน production เราอาจไม่อยาก restore table ทันที แต่ต้องการสร้าง copy ของ old snapshot เพื่อ compare หรือให้ทีม business validate ก่อน


In [33]:
(
    df_known_good_snapshot
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(recovery_copy_table_name)
)

print("Recovery copy table:")
spark.read.table(recovery_copy_table_name).orderBy("customer_id").show(truncate=False)

StatementMeta(, efcf106b-f584-4fbe-abb8-5abf1cab0427, 36, Finished, Available, Finished, False)

Recovery copy table:
+-----------+-------------+------------------------+-------+--------------+-------------+-------------------+
|customer_id|customer_name|email                   |status |customer_score|source_system|updated_at         |
+-----------+-------------+------------------------+-------+--------------+-------------+-------------------+
|C001       |Anan         |anan@example.com        |active |82            |crm          |2026-06-01 09:00:00|
|C002       |Benjawan     |benjawan.new@example.com|active |88            |crm          |2026-06-01 11:00:00|
|C003       |Chai         |chai@example.com        |active |61            |crm          |2026-06-01 11:05:00|
|C004       |Darika       |darika@example.com      |active |91            |crm          |2026-06-01 09:15:00|
|C005       |Ekkarat      |ekkarat@example.com     |active |68            |mobile_app   |2026-06-01 10:00:00|
|C006       |Fah          |fah@example.com         |pending|55            |mobile_app   |2026-06-01

## Quick recap

| Question | Answer |
|---|---|
| `DESCRIBE HISTORY` ใช้ทำอะไร? | ใช้ดู table versions และ operations ที่เกิดขึ้นกับ Delta table |
| Time travel เปลี่ยน current table ไหม? | ไม่เปลี่ยน เป็น read-only query ไปยัง old snapshot |
| `versionAsOf` เหมาะกับอะไร? | อ่าน table ณ version ที่กำหนด โดยระบุ version number ตรง ๆ |
| Restore ต่างจาก time travel ยังไง? | Restore ทำให้ old version กลายเป็น current table state ใหม่ |
| ก่อน restore ควรทำอะไร? | Inspect history, time travel ไปดู old snapshot และ validate ก่อน |
| `VACUUM` เกี่ยวข้องยังไง? | ถ้า `VACUUM` ลบ files ที่ version เก่าต้องใช้ไปแล้ว จะย้อนกลับไปอ่านหรือ restore version นั้นไม่ได้ |


## Exercises


### Exercise 1: Build a compact history summary

สร้าง DataFrame ชื่อ `df_history_summary` เพื่อสรุป version history ของ table

Requirements:

- อ่าน history ของ `target_table_name`
- เลือก columns สำคัญ เช่น `version`, `timestamp`, `operation`, `operationParameters`
- sort จาก version ล่าสุดไปเก่าสุด

Expected result:

- เห็น version ของ `WRITE`, `MERGE`, bad overwrite และ `RESTORE`


In [34]:
df_history_summary = (
    spark.sql(f"DESCRIBE HISTORY {target_table_name}")
    .select(
        "version",
        "timestamp",
        "operation",
        "operationParameters"
    )
    .orderBy(F.col("version").desc())
)

df_history_summary.show(truncate=False)

StatementMeta(, efcf106b-f584-4fbe-abb8-5abf1cab0427, 37, Finished, Available, Finished, False)

+-------+-----------------------+---------------------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|version|timestamp              |operation                        |operationParameters                                                                                                                                                                                                                                  |
+-------+-----------------------+---------------------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|4      |2026-06-02 14:08:15.824|RESTORE                  

### Exercise 2: Compare current table with the known-good snapshot

เปรียบเทียบ current table หลัง restore กับ known-good snapshot ที่อ่านจาก time travel

Requirements:

- group by `status`
- count records ต่อ status
- แสดงผล current summary และ known-good summary

Expected result:

- หลัง restore แล้ว current summary ควรตรงกับ known-good snapshot


In [35]:
df_current_status_summary = (
    spark.read.table(target_table_name)
    .groupBy("status")
    .agg(F.count("*").alias("current_record_count"))
    .orderBy("status")
)

df_known_good_status_summary = (
    df_known_good_snapshot
    .groupBy("status")
    .agg(F.count("*").alias("known_good_record_count"))
    .orderBy("status")
)

print("Current table status summary:")
df_current_status_summary.show()

print("Known-good snapshot status summary:")
df_known_good_status_summary.show()

StatementMeta(, efcf106b-f584-4fbe-abb8-5abf1cab0427, 38, Finished, Available, Finished, False)

Current table status summary:
+-------+--------------------+
| status|current_record_count|
+-------+--------------------+
| active|                   6|
|pending|                   1|
+-------+--------------------+

Known-good snapshot status summary:
+-------+-----------------------+
| status|known_good_record_count|
+-------+-----------------------+
| active|                      6|
|pending|                      1|
+-------+-----------------------+



### Exercise 3: Create a small recovery validation report

สร้าง validation report เพื่อตอบคำถามว่า recovery สำเร็จหรือไม่ในระดับ row count

Requirements:

- นับ records ของ current table
- นับ records ของ known-good snapshot
- สร้าง DataFrame ที่มี columns: `check_name`, `current_value`, `expected_value`, `check_status`
- `check_status` เป็น `PASS` เมื่อจำนวน records เท่ากัน

Expected result:

- ได้ validation report แบบสั้น ๆ ที่ใช้เป็น pattern สำหรับ recovery check ได้


In [36]:
current_table_count = spark.read.table(target_table_name).count()
expected_table_count = df_known_good_snapshot.count()

# Each tuple represents one row in the DataFrame.
# The tuple values must follow the same order as the DataFrame schema or column list.
validation_rows = [
    (
        "row_count_after_restore",
        current_table_count,
        expected_table_count,
        "PASS" if current_table_count == expected_table_count else "FAIL",
    )
]

df_recovery_validation = spark.createDataFrame(
    validation_rows,
    ["check_name", "current_value", "expected_value", "check_status"]
)

df_recovery_validation.show(truncate=False)

StatementMeta(, efcf106b-f584-4fbe-abb8-5abf1cab0427, 40, Finished, Available, Finished, False)

+-----------------------+-------------+--------------+------------+
|check_name             |current_value|expected_value|check_status|
+-----------------------+-------------+--------------+------------+
|row_count_after_restore|7            |7             |PASS        |
+-----------------------+-------------+--------------+------------+



## Common mistakes

1. **คิดว่า time travel คือ rollback**

   Time travel แค่อ่าน old snapshot แบบ read-only ถ้าต้องเปลี่ยน current table state ต้องใช้ restore หรือเขียน recovered snapshot กลับเองอย่างตั้งใจ

2. **Restore โดยไม่ validate old version ก่อน**

   ห้ามเลือก version จากความจำหรือเดาสุ่ม ควรตรวจ history, row count, key records และ business logic ก่อน restore

3. **ลืมว่า bad overwrite ก็เป็น version หนึ่งใน history**

   History จะเห็นทั้ง good และ bad operations ดังนั้นต้องแยกให้ออกว่า version ไหนคือ incident version และ version ไหนคือ known-good version

4. **รัน `VACUUM` แบบ aggressive เกินไป**

   ถ้า old files ถูก cleanup ไปแล้ว จะทำ time travel หรือ restore version เก่าไม่ได้ ควร align retention กับ recovery requirement

5. **ใช้ table history แทน audit log ทั้งหมด**

   Delta history ช่วยมาก แต่ production pipeline ยังควรมี audit/run log ของตัวเอง เช่น run_id, batch_id, source_file, row_count และ status

6. **ทดลอง restore บน production table โดยตรง**

   สำหรับ critical table ควร test ด้วย cloned table หรือ recovery copy ก่อน เพื่อยืนยันว่า restore version ที่เลือกไว้ถูกต้องจริง ก่อนเปลี่ยน current production table


## Expected Output / Success Criteria

เมื่อจบ Day 17 ควรทำได้:

- อธิบายได้ว่า Delta table version เกิดจาก committed changes ใน transaction log
- ใช้ `DESCRIBE HISTORY` เพื่อตรวจ table operations และ versions ได้
- อ่าน old snapshot ด้วย `versionAsOf` หรือ `VERSION AS OF` ได้
- อธิบายได้ว่า time travel เป็น read-only และไม่ใช่ rollback
- จำลอง bad overwrite แล้วใช้ known-good version เพื่อตรวจข้อมูลก่อน incident ได้
- ใช้ `RESTORE TABLE ... TO VERSION AS OF ...` เพื่อ recover demo table ได้
- สร้าง recovery copy table จาก historical snapshot ได้
- อธิบายข้อจำกัดของ time travel / restore ที่เกี่ยวข้องกับ retention และ `VACUUM` ได้


## Final takeaway

Delta history ทำให้ Lakehouse table มี recovery mindset ที่ดีขึ้น เพราะทุก committed change ถูกเก็บเป็น version ให้ตรวจสอบย้อนหลัง และใช้ประกอบการ recovery ได้

> อย่า restore จากความรู้สึก ให้ restore จาก evidence: history, old snapshot, validation และ known-good version

จำ pattern หลักของวันนี้:

- `DESCRIBE HISTORY` เพื่อหา timeline ของ table
- `versionAsOf` / `VERSION AS OF` เพื่อ inspect old snapshot
- validate ก่อน restore เสมอ
- `RESTORE` จะสร้าง current version ใหม่ ไม่ใช่ลบ history เดิม
- ระวัง `VACUUM` เพราะอาจลดความสามารถในการ time travel / recovery


## Optional cleanup

In [ ]:
# spark.sql(f"DROP TABLE IF EXISTS {target_table_name}")
# spark.sql(f"DROP TABLE IF EXISTS {recovery_copy_table_name}")
# spark.catalog.dropTempView("day17_customer_updates_vw")